# Feast with a Ray offline store on KubeRay

This notebook documents the Feast 0.61 contributed Ray offline store and Ray batch engine on a KubeRay-managed cluster. Feast uses Ray for parallel file-based offline computation; KubeRay provides the separate `RayCluster` and `RayJob` resources. The Feast Operator does not create or manage the Ray cluster.

Read this notebook first when following the Feast examples. Then continue with [Feast offline-to-online inference](https://github.com/alauda/aml-docs/tree/master/docs/en/train/guides/feast-offline-to-online-inference.ipynb), which uses the prepared feature data and demonstrates Redis online serving, KServe inference, performance testing, and metrics.

```text
FileSource (Parquet) -> Feast Ray offline store / Ray batch engine -> registry and online store
                                      ^
                              KubeRay RayCluster
```

Keep this notebook together with `assets/feast-with-ray-offline-store/`. From a Workbench terminal, download both paths with a sparse checkout:

```bash
git clone --depth 1 --filter=blob:none --sparse https://github.com/alauda/aml-docs.git
git -C aml-docs sparse-checkout set --no-cone \
  /docs/en/train/guides/feast-with-ray-offline-store.ipynb \
  /docs/en/train/guides/assets/feast-with-ray-offline-store/
cd aml-docs/docs/en/train/guides
```

## Prerequisites

- the Alauda Build of KubeRay Operator is installed;
- an approved runtime image contains `feast[ray]`, Ray, CodeFlare/KubeRay client dependencies, and the object-store connector required by the `FileSource`;
- the runtime can access the S3-compatible object store and its credentials are supplied by a Secret or workload identity; and
- `kubectl`, `bash`, and `curl` are available in the notebook environment.


## Configure Feast and the operator

There are two configuration layers. The Feast project repository's `feature_store.yaml` is read by Feast SDK/CLI jobs and describes the Ray offline store and batch engine. The Kubernetes `FeatureStore` CR is read by the Feast Operator and manages services, Secrets, and ConfigMaps; it does not replace `feature_store.yaml`. Store the Ray offline-store YAML in a Secret and the batch-engine YAML in a ConfigMap, then reference them from the CR.

In the Feast project repository, configure `feature_store.yaml` as follows. This direct-address mode connects to the Ray Client endpoint exposed by the head Service:

```yaml
offline_store:
  type: ray
  storage_path: s3://<bucket>/feast-data
  ray_address: ray://<ray-head-service>:10001

batch_engine:
  type: ray.engine
  max_workers: 8
```

For CodeFlare/KubeRay submission, use `use_kuberay: true` and configure `kuberay_conf.cluster_name`, `kuberay_conf.namespace`, and the API authentication fields instead of `ray_address`. Set explicit Ray CPU and memory limits, and keep the Ray version in the image, `RayCluster`, and Feast dependencies aligned.

For an operator-managed `FeatureStore`, use a data-store Secret and a batch-engine ConfigMap like these examples:

```yaml
apiVersion: v1
kind: Secret
metadata:
  name: feast-ray-store
type: Opaque
stringData:
  ray: |
    type: ray
    storage_path: s3://<bucket>/feast-data
    ray_address: ray://<ray-head-service>:10001
---
apiVersion: v1
kind: ConfigMap
metadata:
  name: feast-ray-engine
data:
  config: |
    type: ray.engine
    max_workers: 8
---
apiVersion: feast.dev/v1
kind: FeatureStore
metadata:
  name: feast-ray
spec:
  feastProject: feast_demo
  services:
    offlineStore:
      persistence:
        store:
          type: ray
          secretRef:
            name: feast-ray-store
  batchEngine:
    configMapRef:
      name: feast-ray-engine
```

Keep credentials out of the notebook and ConfigMap. Use a Secret, workload identity, or an external-secret integration for object-store and Ray API authentication.

The Ray offline store supports `FileSource` data and has no direct SQL query interface. It does not write an online store by itself; run Feast materialization with the Ray batch engine after offline data is available. See the [Feast Ray offline-store reference](https://docs.feast.dev/v0.61-branch/reference/offline-stores/ray) and [Ray compute-engine reference](https://docs.feast.dev/v0.61-branch/reference/compute-engine/ray).

## Production-scale data and compute

For large datasets, treat object storage and partition design as the primary performance boundary. Keep immutable Parquet in an S3-compatible bucket, partition by an event-date column used by the retrieval window, and use a consistent schema and compression codec. Avoid millions of tiny files: compact partitions into files large enough for sequential reads, but small enough to distribute across Ray workers.

Use time-bounded and incremental retrievals instead of repeatedly scanning the full history. Apply entity and timestamp filters early, keep only the columns needed by the feature view, and materialize only the new event-time interval. Validate that the FileSource paths and timestamps are timezone-consistent before scaling out.

Size the `RayCluster` for the largest join, not the average request. Set CPU and memory requests/limits on head and workers, reserve headroom for the Ray dashboard and object store, and scale workers horizontally for independent partitions. Use autoscaling only after setting upper bounds and testing shuffle-heavy joins; spilling to a fast local or ephemeral volume is preferable to exhausting pod memory.

For production runs, make jobs retryable and idempotent. Write results to a versioned output prefix, publish a completion marker only after all partitions succeed, and avoid mutating a prefix that another materialization run can read. Set Ray and Kubernetes job timeouts, retain failed driver logs, and use a queue such as Kueue when multiple teams share the cluster.

Benchmark with production-shaped row counts, feature widths, skew, and concurrent materializations. Record wall-clock duration, input/output bytes, peak worker memory, spill volume, retries, and the fraction of partitions that fail. A successful sample run validates wiring only; it is not evidence of capacity or an SLA.

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_RAY_ASSET_DIR:-assets/feast-with-ray-offline-store}"
: "${FEAST_RAY_IMAGE:?Set FEAST_RAY_IMAGE to an approved Feast/Ray runtime image}"
test -f "$ASSET_DIR/raycluster.yaml"
test -f "$ASSET_DIR/rayjob.yaml"
kubectl get crd featurestores.feast.dev
kubectl get crd rayclusters.ray.io
kubectl get crd rayjobs.ray.io


## Deploy a KubeRay cluster

The supplied manifest is a minimal starting point with one worker. Replace the placeholder image with an approved registry image that contains Feast and Ray, then tune replicas, autoscaling, resources, security, and storage for the production workload. Set `FEAST_RAY_CLUSTER_NAME` to deploy more than one cluster or to use an existing naming convention.

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_RAY_ASSET_DIR:-assets/feast-with-ray-offline-store}"
NAMESPACE="${FEAST_RAY_NAMESPACE:-mlops-demo-e2e}"
CLUSTER_NAME="${FEAST_RAY_CLUSTER_NAME:-feast-ray-cluster}"
sed -e "s#<approved-registry>/mlops/feast-ray-runtime:<tag>#$FEAST_RAY_IMAGE#g" -e "s#feast-ray-cluster#$CLUSTER_NAME#g" "$ASSET_DIR/raycluster.yaml" | kubectl -n "$NAMESPACE" apply -f -
for _ in $(seq 1 120); do
  state="$(kubectl -n "$NAMESPACE" get raycluster "$CLUSTER_NAME" -o jsonpath='{.status.state}' 2>/dev/null || true)"
  echo "RayCluster state=${state:-Pending}"
  [ "$state" = Running ] && break
  sleep 5
done
[ "${state:-}" = Running ] || { kubectl -n "$NAMESPACE" describe raycluster "$CLUSTER_NAME"; exit 1; }
kubectl -n "$NAMESPACE" get raycluster "$CLUSTER_NAME" -o wide


## Run a Feast/Ray job

The supplied job is a template: it imports Feast and Ray, connects to the selected Ray cluster, and prints the Ray node count. Replace its entrypoint with the project-specific Parquet `FileSource`, `get_historical_features`, or materialization command. For production, submit one job per bounded event-time window, use durable versioned output paths, configure retry and timeout policies, and retain the driver logs and job status as run evidence. Set `FEAST_RAY_JOB_NAME` and `FEAST_RAY_CLUSTER_NAME` for your deployment.

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${FEAST_RAY_ASSET_DIR:-assets/feast-with-ray-offline-store}"
NAMESPACE="${FEAST_RAY_NAMESPACE:-mlops-demo-e2e}"
CLUSTER_NAME="${FEAST_RAY_CLUSTER_NAME:-feast-ray-cluster}"
JOB_NAME="${FEAST_RAY_JOB_NAME:-feast-ray-job}"
sed -e "s#feast-ray-cluster#$CLUSTER_NAME#g" -e "s#feast-ray-job#$JOB_NAME#g" "$ASSET_DIR/rayjob.yaml" | kubectl -n "$NAMESPACE" apply -f -
for _ in $(seq 1 120); do
  job_status="$(kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o jsonpath='{.status.jobStatus}' 2>/dev/null || true)"
  echo "RayJob status=${job_status:-Pending}"
  [ "$job_status" = SUCCEEDED ] && break
  [ "$job_status" = FAILED ] && { kubectl -n "$NAMESPACE" describe rayjob "$JOB_NAME"; exit 1; }
  sleep 5
done
[ "${job_status:-}" = SUCCEEDED ] || { kubectl -n "$NAMESPACE" describe rayjob "$JOB_NAME"; exit 1; }
kubectl -n "$NAMESPACE" logs -l ray.io/job-name="$JOB_NAME" --tail=100


In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${FEAST_RAY_NAMESPACE:-mlops-demo-e2e}"
CLUSTER_NAME="${FEAST_RAY_CLUSTER_NAME:-feast-ray-cluster}"
JOB_NAME="${FEAST_RAY_JOB_NAME:-feast-ray-job}"
kubectl -n "$NAMESPACE" delete rayjob "$JOB_NAME" --ignore-not-found
kubectl -n "$NAMESPACE" delete raycluster "$CLUSTER_NAME" --ignore-not-found
